# Data Preprocessing & Feature Extraction

# 本 notebook 实现 README.md 中建模思路所需的数据预处理与特征提取：
# - 读取原始 CSV
# - 清洗缺失/占位值（`N/A`, 空字符串 等）
# - 计算每位选手每周的评委总分 J_{i,t}
# - 将数据转为 long 格式（每行 = 选手-周）并确定选手在赛季中的最后活跃周
# - 计算观测特征：P^J（评委百分比）、R^J（评委名次）、历史统计量、得分变化等
# - 保存处理后的特征文件用于后续建模

In [1]:
# Imports
import re
import os
import numpy as np
import pandas as pd
from pathlib import Path

WORKDIR = Path('.')
RAW_CSV = WORKDIR / '2026_MCM_Problem_C_Data.csv'
OUT_CSV = WORKDIR / 'processed_features.csv'
OUT_PARQUET = WORKDIR / 'processed_features.parquet'
print('raw csv ->', RAW_CSV.resolve())

raw csv -> E:\Documents\Mathematical-Modelling\2026_MCM_Problem_C_Data.csv


In [2]:
# Load data and basic cleaning
df = pd.read_csv(RAW_CSV)
# Normalize string columns and replace 'N/A' with NaN
df = df.replace('N/A', np.nan)
df.columns = [c.strip() for c in df.columns]
if 'celebrity_name' in df.columns:
    df['celebrity_name'] = df['celebrity_name'].astype(str).str.strip()
# strip trailing/leading spaces in textual fields
for c in ['ballroom_partner','celebrity_industry','celebrity_homestate','celebrity_homecountry/region','results']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace({'nan': None})

df.head(3)

,celebrity_name,ballroom_partner,celebrity_industry,celebrity_homestate,celebrity_homecountry/region,celebrity_age_during_season,season,results,placement,week1_judge1_score,...,week9_judge3_score,week9_judge4_score,week10_judge1_score,week10_judge2_score,week10_judge3_score,week10_judge4_score,week11_judge1_score,week11_judge2_score,week11_judge3_score,week11_judge4_score
0,John O'Hurley,Charlotte Jorgensen,Actor/Actress,Maine,United States,50,1,2nd Place,2,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kelly Monaco,Alec Mazo,Actor/Actress,Pennsylvania,United States,29,1,1st Place,1,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Evander Holyfield,Edyta Sliwinska,Athlete,Alabama,United States,42,1,Eliminated Week 3,5,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df['celebrity_industry'].value_counts()

celebrity_industry
Actor/Actress               128
Athlete                      95
TV Personality               67
Singer/Rapper                61
Model                        17
Comedian                     12
Social Media Personality      8
Entrepreneur                  4
Radio Personality             4
Racing Driver                 4
Politician                    3
News Anchor                   3
Sports Broadcaster            2
Beauty Pagent                 1
Magician                      1
Astronaut                     1
Fashion Designer              1
Motivational Speaker          1
Military                      1
Journalist                    1
Musician                      1
Producer                      1
Fitness Instructor            1
Con artist                    1
Conservationist               1
Social media personality      1
Name: count, dtype: int64

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MultiLabelBinarizer

WORKDIR = Path('.')
PROC_CSV = WORKDIR / 'processed_features.csv'
OUT_TAGGED = WORKDIR / 'processed_features_tagged.csv'
OUT_MODEL = WORKDIR / 'final_features_for_model.csv'

print('Reading', PROC_CSV)
df = pd.read_csv(PROC_CSV)

# Load celebrities to get original notability text (if present)
celebs = pd.read_csv(WORKDIR / 'celebrities.csv')
celebs.columns = [c.strip() for c in celebs.columns]
celebs['celebrity'] = celebs['celebrity'].astype(str).str.strip()
celebs['notability_text'] = celebs.get('notability', '').astype(str).str.strip()

# Align notability_text into df if missing
if 'notability_text' not in df.columns:
    df['celebrity_name_clean'] = df['celebrity_name'].astype(str).str.strip()
    merge = df.merge(celebs[['celebrity','notability_text']], left_on='celebrity_name_clean', right_on='celebrity', how='left')
    merge['notability_text'] = merge['notability_text'].fillna('')
else:
    merge = df.copy()
    merge['notability_text'] = merge['notability_text'].fillna('')

# parse notability into categorical labels per user's instruction
def parse_notability_tags(text):
    s = str(text).lower()
    tags = set()
    if any(k in s for k in ['olymp','olympic','olympics']):
        tags.add('Olympic')
    if 'disney' in s:
        tags.add('Disney')
    if any(k in s for k in ['host','presenter','talk show','talk-show','talkshow']):
        tags.add('Host')
    if any(k in s for k in ['singer','musician','song','rapper']):
        tags.add('Singer')
    if any(k in s for k in ['actor','actress','film','television','movie','soap']):
        tags.add('Actor')
    if any(k in s for k in ['athlete','box','nfl','nba','mlb','ufc','gymnast','skater','swimmer','luger','racer','sprinter','runner']):
        tags.add('Athlete')
    if any(k in s for k in ['model','supermodel','miss','playboy']):
        tags.add('Model')
    if any(k in s for k in ['reality','bachelor','jersey','housewives','realit','youtube','social media']):
        tags.add('Reality')
    if any(k in s for k in ['journalist','anchor','news','reporter']):
        tags.add('Journalist')
    if any(k in s for k in ['judge','shark tank','judge panel']):
        tags.add('Judge')
    if len(tags) == 0:
        tags.add('Other')
    return list(tags)

merge['notability_tags'] = merge['notability_text'].apply(parse_notability_tags)
mlb = MultiLabelBinarizer()
notab_mat = mlb.fit_transform(merge['notability_tags'])
notab_cols = [f'notab_tag_{t.replace(" ","_")}' for t in mlb.classes_]
notab_df = pd.DataFrame(notab_mat, columns=notab_cols, index=merge.index)

# One-hot encode categorical columns (by kind) as the user requested
# Use full one-hot; replace NaN/empty with 'Unknown'
def full_one_hot(series, prefix=''):
    s = series.fillna('Unknown').astype(str).str.strip()
    s = s.replace({'': 'Unknown', 'nan': 'Unknown', 'None': 'Unknown'})
    return pd.get_dummies(s, prefix=prefix)

industry_df = full_one_hot(merge.get('celebrity_industry', pd.Series()), prefix='industry')
state_df = full_one_hot(merge.get('celebrity_homestate', pd.Series()), prefix='state')
country_df = full_one_hot(merge.get('celebrity_homecountry/region', pd.Series()), prefix='country')

# Concatenate engineered categorical features (include notability tags)
features = pd.concat([industry_df.reset_index(drop=True), state_df.reset_index(drop=True), country_df.reset_index(drop=True), notab_df.reset_index(drop=True)], axis=1)
features = features.loc[:, (features.sum(axis=0) != 0)]

# PCA (retain 95% variance)
scaler = StandardScaler()
X = features.fillna(0).values
if X.shape[1] == 0:
    raise RuntimeError('No features to PCA on')
Xs = scaler.fit_transform(X)
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(Xs)
print('PCA shape:', X_pca.shape)
print('Explained variance sum:', pca.explained_variance_ratio_.sum())

# attach PCA cols
pca_cols = [f'pca_tag_{i+1}' for i in range(X_pca.shape[1])]
for i,col in enumerate(pca_cols):
    merge[col] = X_pca[:, i]

# also attach notability tag columns and categorical dummies for traceability
merge = pd.concat([merge.reset_index(drop=True), notab_df.reset_index(drop=True), industry_df.reset_index(drop=True), state_df.reset_index(drop=True), country_df.reset_index(drop=True)], axis=1)

merge.to_csv(OUT_TAGGED, index=False)
print('Saved tagged processed features to', OUT_TAGGED)

# reduce to 5 components for modeling and combine with key metadata
n_keep = min(5, X_pca.shape[1])
from sklearn.decomposition import PCA as PCA2
pca5 = PCA2(n_components=n_keep, random_state=42)
X5 = pca5.fit_transform(Xs)
df_model = pd.DataFrame(X5, columns=[f'pca5_{i+1}' for i in range(n_keep)])
for col in ['celebrity_name','season','placement','celebrity_industry','celebrity_homestate','celebrity_homecountry/region']:
    if col in merge.columns:
        df_model[col] = merge[col].values
# include notability tag count and also the explicit notability tag columns
if len(notab_cols) > 0:
    df_model['notability_tag_count'] = notab_df.sum(axis=1).values
    for c in notab_cols:
        df_model[c] = notab_df[c].values

df_model.to_csv(OUT_MODEL, index=False)
print('Saved final model features to', OUT_MODEL)
print('Done')


Reading processed_features.csv
PCA shape: (421, 89)
Explained variance sum: 0.9544013677977767
Saved tagged processed features to processed_features_tagged.csv
Saved final model features to final_features_for_model.csv
Done


In [5]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

WORKDIR = Path('.')
IN_CSV = WORKDIR / 'processed_features_tagged.csv'
OUT_DIR = WORKDIR

if not IN_CSV.exists():
    print('Error: processed_features_tagged.csv not found at', IN_CSV)
    sys.exit(1)

print('Reading', IN_CSV)
df = pd.read_csv(IN_CSV)

# Identify engineered feature columns (categorical dummies + notability tags)
feat_prefixes = ('industry_', 'state_', 'country_', 'notab_tag_')
feat_cols = [c for c in df.columns if c.startswith(feat_prefixes)]
if len(feat_cols) == 0:
    # fallback: any column that looks like a one-hot (contains '_' and low cardinality)
    guess = [c for c in df.columns if ('_' in c and df[c].nunique() <= 2)]
    feat_cols = guess

print('Using', len(feat_cols), 'feature columns for PCA')
if len(feat_cols) == 0:
    print('No suitable feature columns found. Exiting.')
    sys.exit(1)

X = df[feat_cols].fillna(0).values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

n_comp = min(50, Xs.shape[1])
pca = PCA(n_components=n_comp, random_state=42)
Xp = pca.fit_transform(Xs)
explained = pca.explained_variance_ratio_
cum = np.cumsum(explained)

# Scree plot
plt.figure(figsize=(8,4))
plt.plot(np.arange(1, len(explained)+1), explained, '-o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Scree')
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_scree.png')
plt.close()

# Cumulative
plt.figure(figsize=(8,4))
plt.plot(np.arange(1, len(cum)+1), cum, '-o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Explained Variance')
plt.axhline(0.95, color='red', linestyle='--', label='0.95')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_cumulative.png')
plt.close()

# Top loadings per component (first 5)
components = pca.components_  # shape (n_comp, n_features)
cols = feat_cols
rows = []
num_show = 10
n_show = min(5, components.shape[0])
for i in range(n_show):
    comp = components[i]
    # top positive
    pos_idx = np.argsort(comp)[-num_show:][::-1]
    neg_idx = np.argsort(comp)[:num_show]
    for rank, idx in enumerate(pos_idx, start=1):
        rows.append({'component': f'PC{i+1}', 'side': 'pos', 'rank': rank, 'feature': cols[idx], 'loading': float(comp[idx])})
    for rank, idx in enumerate(neg_idx, start=1):
        rows.append({'component': f'PC{i+1}', 'side': 'neg', 'rank': rank, 'feature': cols[idx], 'loading': float(comp[idx])})

out_df = pd.DataFrame(rows)
out_csv = OUT_DIR / 'pca_top_loadings.csv'
out_df.to_csv(out_csv, index=False)
print('Saved top loadings to', out_csv)

# Scatter PC1 vs PC2 colored by placement (if exists)
pc = Xp.shape[1]
pc1 = 0
pc2 = 1 if pc>1 else 0
placement = None
if 'placement' in df.columns:
    # try convert to numeric
    try:
        placement = pd.to_numeric(df['placement'], errors='coerce')
    except Exception:
        placement = None

plt.figure(figsize=(6,6))
if placement is not None:
    sc = plt.scatter(Xp[:,pc1], Xp[:,pc2], c=placement.fillna(-1), cmap='viridis', s=30)
    plt.colorbar(sc, label='placement (numeric or -1)')
else:
    plt.scatter(Xp[:,pc1], Xp[:,pc2], s=30)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PC1 vs PC2')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_pc1_pc2.png')
plt.close()
print('Saved PC1-vs-PC2 scatter to pca_pc1_pc2.png')

# Optional: 2D t-SNE for visualization (may be slow)
try:
    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    Z = tsne.fit_transform(Xs)
    plt.figure(figsize=(6,6))
    if placement is not None:
        sc = plt.scatter(Z[:,0], Z[:,1], c=placement.fillna(-1), cmap='viridis', s=25)
        plt.colorbar(sc, label='placement (numeric or -1)')
    else:
        plt.scatter(Z[:,0], Z[:,1], s=25)
    plt.title('t-SNE 2D')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'tsne_2d.png')
    plt.close()
    print('Saved t-SNE projection to tsne_2d.png')
except Exception as e:
    print('t-SNE failed:', e)

# Print brief summary to stdout: explained variance for first 10 components and top features
n_summary = min(10, len(explained))
print('\nExplained variance (first {} components):'.format(n_summary))
for i in range(n_summary):
    print(f'PC{i+1}: {explained[i]:.4f}  cumulative: {cum[i]:.4f}')

print('\nTop features per first {} PCs (pos / neg):'.format(n_show))
for i in range(n_show):
    comp = components[i]
    pos_idx = np.argsort(comp)[-num_show:][::-1]
    neg_idx = np.argsort(comp)[:num_show]
    print(f'\nPC{i+1} (+):')
    for idx in pos_idx:
        print(f'  {cols[idx]}: {comp[idx]:.4f}')
    print(f'PC{i+1} (-):')
    for idx in neg_idx:
        print(f'  {cols[idx]}: {comp[idx]:.4f}')

print('\nAll outputs saved to', OUT_DIR)


Reading processed_features_tagged.csv
Using 109 feature columns for PCA
Saved top loadings to pca_top_loadings.csv
Saved PC1-vs-PC2 scatter to pca_pc1_pc2.png
Saved t-SNE projection to tsne_2d.png

Explained variance (first 10 components):
PC1: 0.0314  cumulative: 0.0314
PC2: 0.0279  cumulative: 0.0593
PC3: 0.0220  cumulative: 0.0813
PC4: 0.0213  cumulative: 0.1026
PC5: 0.0205  cumulative: 0.1231
PC6: 0.0194  cumulative: 0.1426
PC7: 0.0178  cumulative: 0.1604
PC8: 0.0175  cumulative: 0.1779
PC9: 0.0158  cumulative: 0.1937
PC10: 0.0158  cumulative: 0.2095

Top features per first 5 PCs (pos / neg):

PC1 (+):
  country_United States: 0.4432
  industry_Athlete: 0.3076
  notab_tag_Athlete: 0.3039
  notab_tag_Olympic: 0.2335
  state_Iowa: 0.0854
  state_Washington: 0.0832
  state_Massachusetts: 0.0772
  state_Florida: 0.0700
  state_West Virginia: 0.0581
  state_Kansas: 0.0578
PC1 (-):
  state_Unknown: -0.4432
  country_England: -0.1938
  notab_tag_Actor: -0.1726
  country_Canada: -0.1685
  

In [6]:
# PCA sweep: 尝试 1 到 7 维并保存解释方差与 PC1-vs-PC2 图（当维度>=2 时）
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

WORKDIR = Path('.')
IN_CSV = WORKDIR / 'processed_features_tagged.csv'
df = pd.read_csv(IN_CSV)
feat_prefixes = ('industry_', 'state_', 'country_', 'notab_tag_')
feat_cols = [c for c in df.columns if c.startswith(feat_prefixes)]
if len(feat_cols) == 0:
    raise RuntimeError('No feature columns found for PCA sweep')
X = df[feat_cols].fillna(0).values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
for n in range(1,8):
    pca = PCA(n_components=n, random_state=42)
    Xp = pca.fit_transform(Xs)
    explained = pca.explained_variance_ratio_
    print(f'PCA n={n}, explained sum={explained.sum():.4f}')
    out_exp = WORKDIR / f'pca_n{n}_explained.csv'
    pd.DataFrame({'pc': list(range(1, len(explained)+1)), 'explained_variance_ratio': explained}).to_csv(out_exp, index=False)
    if n >= 2:
        plt.figure(figsize=(6,6))
        plt.scatter(Xp[:,0], Xp[:,1], s=30)
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.title(f'PC1 vs PC2 (n={n})')
        plt.tight_layout()
        plt.savefig(WORKDIR / f'pca_pc1_pc2_n{n}.png')
        plt.close()
print('PCA sweep finished; saved CSVs and plots to workspace')

PCA n=1, explained sum=0.0314
PCA n=2, explained sum=0.0593
PCA n=3, explained sum=0.0813
PCA n=4, explained sum=0.1026
PCA n=5, explained sum=0.1231
PCA n=6, explained sum=0.1426
PCA n=7, explained sum=0.1604
PCA sweep finished; saved CSVs and plots to workspace


In [7]:
# ...existing code...
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

WORKDIR = Path('.')
IN_CSV = WORKDIR / 'processed_features_tagged.csv'
OUT_CSV = WORKDIR / 'pca_n2_feature_correlations.csv'
OUT_TOP = WORKDIR / 'pca_n2_top_features.csv'

df = pd.read_csv(IN_CSV)

# detect engineered feature cols (same heuristic as earlier)
feat_prefixes = ('industry_', 'state_', 'country_', 'notab_tag_')
feat_cols = [c for c in df.columns if c.startswith(feat_prefixes)]
if len(feat_cols) == 0:
    feat_cols = [c for c in df.columns if ('_' in c and df[c].nunique() <= 2)]

if len(feat_cols) == 0:
    raise RuntimeError('No feature columns found for correlation analysis')

# prepare scaled feature matrix and PCA with n=2
X = df[feat_cols].fillna(0).values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
pca = PCA(n_components=2, random_state=42)
Xp = pca.fit_transform(Xs)  # shape (n_samples, 2)

# compute Pearson correlations between each (scaled) feature and each PC score
corrs = []
for i, feat in enumerate(feat_cols):
    s = Xs[:, i]
    r1 = pd.Series(s).corr(pd.Series(Xp[:, 0]))
    r2 = pd.Series(s).corr(pd.Series(Xp[:, 1]))
    corrs.append({'feature': feat, 'pc1_corr': float(r1), 'pc2_corr': float(r2)})

corr_df = pd.DataFrame(corrs).set_index('feature')
corr_df.to_csv(OUT_CSV)

# pick top 10 positive and top 10 negative for each PC
def top_lists(corr_series, topk=10):
    pos = corr_series.sort_values(ascending=False).head(topk)
    neg = corr_series.sort_values(ascending=True).head(topk)
    return pos, neg

pc1_pos, pc1_neg = top_lists(corr_df['pc1_corr'], 10)
pc2_pos, pc2_neg = top_lists(corr_df['pc2_corr'], 10)

# aggregate and save
rows = []
for rank, (f,v) in enumerate(pc1_pos.items(), 1):
    rows.append({'pc':'PC1','side':'pos','rank':rank,'feature':f,'corr':v})
for rank, (f,v) in enumerate(pc1_neg.items(), 1):
    rows.append({'pc':'PC1','side':'neg','rank':rank,'feature':f,'corr':v})
for rank, (f,v) in enumerate(pc2_pos.items(), 1):
    rows.append({'pc':'PC2','side':'pos','rank':rank,'feature':f,'corr':v})
for rank, (f,v) in enumerate(pc2_neg.items(), 1):
    rows.append({'pc':'PC2','side':'neg','rank':rank,'feature':f,'corr':v})

pd.DataFrame(rows).to_csv(OUT_TOP, index=False)

# print brief summary
print('Explained variance (PC1,PC2):', pca.explained_variance_ratio_.tolist())
print('\nTop PC1 (+):\n', pc1_pos)
print('\nTop PC1 (-):\n', pc1_neg)
print('\nTop PC2 (+):\n', pc2_pos)
print('\nTop PC2 (-):\n', pc2_neg)
# ...existing code...

Explained variance (PC1,PC2): [0.03138056626301245, 0.02792090605957481]

Top PC1 (+):
 feature
country_United States    0.819616
industry_Athlete         0.568883
notab_tag_Athlete        0.562123
notab_tag_Olympic        0.431825
state_Iowa               0.157903
state_Washington         0.153918
state_Massachusetts      0.142805
state_Florida            0.129449
state_West Virginia      0.107495
state_Kansas             0.106887
Name: pc1_corr, dtype: float64

Top PC1 (-):
 feature
state_Unknown             -0.819616
country_England           -0.358496
notab_tag_Actor           -0.319209
country_Canada            -0.311605
industry_Model            -0.300032
notab_tag_Model           -0.262014
country_Australia         -0.214906
industry_Actor/Actress    -0.192350
industry_TV Personality   -0.187843
country_Czechoslovakia    -0.180711
Name: pc1_corr, dtype: float64

Top PC2 (+):
 feature
industry_Athlete          0.645224
notab_tag_Athlete         0.582710
state_Unknown             